# Keyword x colour overview

Run `mtg-analysis fetch` and `mtg-analysis build` first — this notebook only reads the
normalized parquet tables.

Every rate here is reported with the denominator behind it, and every share states the
weighting scheme it used. `fractional` splits a multicolour card evenly across its colours
(shares sum to 100%); `inclusive` counts it fully for each colour ("any card touching red").

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from mtg_analysis.analysis.db import get_connection
from mtg_analysis.analysis.design_volume import design_volume, plot_design_volume
from mtg_analysis.analysis.heatmap import (
    keyword_color_matrix,
    pivot_matrix,
    plot_keyword_color_heatmap,
)
from mtg_analysis.analysis.timeseries import keyword_timeseries, plot_keyword_timeseries
from mtg_analysis.config import load_config
from mtg_analysis.metrics.core import color_share, penetration_rate, trend_report

config = load_config("../config/config.yaml")
con = get_connection("../" / config.paths.processed_dir, config.periods)
con.execute("SELECT COUNT(*) AS cards FROM cards").pl()

## 1. Keyword x colour heatmap (deliverable 1)

In [ ]:
matrix = keyword_color_matrix(con, top_n=20)
plot_keyword_color_heatmap(matrix, value="share")
plt.show()
pivot_matrix(matrix, value="raw_count")

## 2. One keyword's headline numbers

Both weightings are printed side by side rather than silently picking one.

In [ ]:
keyword, color = "Double strike", "R"
for weighting in ("fractional", "inclusive"):
    share = color_share(con, keyword, color, weighting=weighting)
    rate = penetration_rate(con, keyword, color, weighting=weighting)
    print(f"{weighting:>10}: {color} holds {share:.1%} of {keyword}; "
          f"{rate:.2%} of {color} cards carry it")

## 3. Trend over sets (deliverable 2)

`trend_report` returns raw count, colour share and penetration rate together — they can
move in opposite directions, and reading one without the others is the main way to get a
claim wrong here.

In [ ]:
trend_report(con, "Haste", "R").tail(10)

In [ ]:
series = keyword_timeseries(con, "Haste", colors=["W", "U", "B", "R", "G"])
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_keyword_timeseries(series, metric="penetration_rate", ax=axes[0])
plot_keyword_timeseries(series, metric="color_share", ax=axes[1])
fig.tight_layout()
plt.show()

## 4. Design volume context (deliverable 8)

The denominator behind every rate above. Always show this next to a raw-count chart.

In [ ]:
volume = design_volume(con)
plot_design_volume(volume)
plt.show()
volume.tail()

## Smoothing noisy sets

Small sets make per-set rates jumpy. Group consecutive sets without rebuilding anything:

In [ ]:
from mtg_analysis.analysis.db import set_period_config
from mtg_analysis.config import PeriodGroupConfig

set_period_config(con, PeriodGroupConfig(mode="rolling_sets", group_size=5))
plot_keyword_timeseries(keyword_timeseries(con, "Haste", colors=["R", "W"]))
plt.show()
set_period_config(con, config.periods)